# Drowsiness Detection — Part A: CNN Baseline
## Driver Monitoring System

**Purpose:** This CNN is a *baseline only* — it exists to demonstrate the limitations of single-frame classification and motivate our temporal PERCLOS-based final system (Part B).
.

## 1. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split

import cv2
from PIL import Image
import glob
import random
from pathlib import Path

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Dataset Paths

In [ ]:
# ── Dataset roots ────────────────────────────────────────────────────────────
MRL_ROOT  = Path('/kaggle/input/datasets/akashshingha850/mrl-eye-dataset/data')
YAWN_ROOT = Path('/kaggle/input/datasets/davidvazquezcic/yawn-dataset')
NTHU_ROOT = Path('/kaggle/input/datasets/samymesbah/nthu-dataset-ddd-multi-class')

# ── Image sizes ───────────────────────────────────────────────────────────────
EYE_SIZE   = (64, 64)   # grayscale
MOUTH_SIZE = (96, 96)   # RGB
BATCH_SIZE = 32
EPOCHS     = 25

# ── Output dir ────────────────────────────────────────────────────────────────
OUT = Path('/kaggle/working/cnn_outputs')
OUT.mkdir(exist_ok=True)

print('Paths configured.')

## 3. Data Loading Utilities

In [ ]:
def load_images_from_folder(folder, label, size, color_mode='grayscale', limit=None):
    """
    Load images from a folder, resize, normalize to [0,1].
    Returns (images_array, labels_array).
    """
    images, labels = [], []
    exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp')
    files = []
    for ext in exts:
        files.extend(glob.glob(str(folder / '**' / ext), recursive=True))

    if limit:
        random.shuffle(files)
        files = files[:limit]

    for fp in files:
        try:
            if color_mode == 'grayscale':
                img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, size)
                img = img.astype(np.float32) / 255.0
                img = img[..., np.newaxis]  # (H, W, 1)
            else:
                img = cv2.imread(fp, cv2.IMREAD_COLOR)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, size)
                img = img.astype(np.float32) / 255.0
            images.append(img)
            labels.append(label)
        except Exception as e:
            pass  # skip corrupt files

    return np.array(images), np.array(labels)


def show_sample_grid(X, y, class_names, n=16, title='Sample Images'):
    """Display a grid of sample images."""
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    idx = np.random.choice(len(X), n, replace=False)
    for ax, i in zip(axes.flat, idx):
        img = X[i]
        if img.shape[-1] == 1:
            ax.imshow(img.squeeze(), cmap='gray')
        else:
            ax.imshow(img)
        ax.set_title(class_names[y[i]], fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUT / f'{title.replace(" ","_")}.png', dpi=120)
    plt.show()


print('Utilities defined.')

## 4. Load Eye Dataset (MRL)

In [ ]:
# ── MRL Eye Dataset ───────────────────────────────────────────────────────────
# Class 0: Open Eye
# Class 1: Closed Eye

# Adjust subfolder names to match actual dataset structure
# Common MRL layout: data/OpenEye/ and data/ClosedEye/
open_eye_dir   = MRL_ROOT / 'OpenEye'
closed_eye_dir = MRL_ROOT / 'ClosedEye'

# Fallback: try alternate naming
if not open_eye_dir.exists():
    # Try finding any folder with 'open' in name
    candidates = [d for d in MRL_ROOT.iterdir() if d.is_dir()]
    print('Available folders in MRL root:', [c.name for c in candidates])
    open_eye_dir   = next((d for d in candidates if 'open' in d.name.lower()), candidates[0])
    closed_eye_dir = next((d for d in candidates if 'close' in d.name.lower()), candidates[1])

print(f'Open eye dir : {open_eye_dir}')
print(f'Closed eye dir: {closed_eye_dir}')

LIMIT = 3000  # per class — balance dataset

X_open,   y_open   = load_images_from_folder(open_eye_dir,   0, EYE_SIZE, 'grayscale', LIMIT)
X_closed, y_closed = load_images_from_folder(closed_eye_dir, 1, EYE_SIZE, 'grayscale', LIMIT)

X_eye = np.concatenate([X_open, X_closed], axis=0)
y_eye = np.concatenate([y_open, y_closed], axis=0)

print(f'Open eyes  : {len(X_open)}')
print(f'Closed eyes: {len(X_closed)}')
print(f'Total eye samples: {len(X_eye)}')
print(f'Image shape: {X_eye[0].shape}')

In [ ]:
show_sample_grid(X_eye, y_eye, ['Open Eye', 'Closed Eye'], title='MRL Eye Dataset Samples')

## 5. Load Yawn Dataset

In [ ]:
# ── Yawn Dataset ──────────────────────────────────────────────────────────────
# Class 0: no_yawn
# Class 1: yawn

yawn_dir    = YAWN_ROOT / 'yawn'
no_yawn_dir = YAWN_ROOT / 'no_yawn'

if not yawn_dir.exists():
    candidates = [d for d in YAWN_ROOT.iterdir() if d.is_dir()]
    print('Available folders in YAWN root:', [c.name for c in candidates])
    yawn_dir    = next((d for d in candidates if 'yawn' in d.name.lower() and 'no' not in d.name.lower()), candidates[0])
    no_yawn_dir = next((d for d in candidates if 'no' in d.name.lower()), candidates[1])

print(f'Yawn dir   : {yawn_dir}')
print(f'No-yawn dir: {no_yawn_dir}')

X_yawn,    y_yawn    = load_images_from_folder(yawn_dir,    1, MOUTH_SIZE, 'rgb', LIMIT)
X_no_yawn, y_no_yawn = load_images_from_folder(no_yawn_dir, 0, MOUTH_SIZE, 'rgb', LIMIT)

X_mouth = np.concatenate([X_yawn, X_no_yawn], axis=0)
y_mouth = np.concatenate([y_yawn, y_no_yawn], axis=0)

print(f'Yawn samples   : {len(X_yawn)}')
print(f'No-yawn samples: {len(X_no_yawn)}')
print(f'Total mouth samples: {len(X_mouth)}')

In [ ]:
show_sample_grid(X_mouth, y_mouth, ['No Yawn', 'Yawn'], title='Yawn Dataset Samples')

## 6. Train/Val/Test Splits

In [ ]:
def split_dataset(X, y, val_size=0.15, test_size=0.15, seed=SEED):
    """80/15/15 stratified split."""
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=(val_size + test_size), stratify=y, random_state=seed
    )
    ratio = test_size / (val_size + test_size)
    X_val, X_te, y_val, y_te = train_test_split(
        X_tmp, y_tmp, test_size=ratio, stratify=y_tmp, random_state=seed
    )
    return X_tr, X_val, X_te, y_tr, y_val, y_te


# Eye splits
X_eye_tr, X_eye_val, X_eye_te, y_eye_tr, y_eye_val, y_eye_te = split_dataset(X_eye, y_eye)

# Mouth splits
X_mo_tr, X_mo_val, X_mo_te, y_mo_tr, y_mo_val, y_mo_te = split_dataset(X_mouth, y_mouth)

print('Eye dataset splits:')
print(f'  Train: {len(X_eye_tr)} | Val: {len(X_eye_val)} | Test: {len(X_eye_te)}')
print('\nMouth dataset splits:')
print(f'  Train: {len(X_mo_tr)} | Val: {len(X_mo_val)} | Test: {len(X_mo_te)}')

## 7. CNN Architecture

In [ ]:
def build_eye_cnn(input_shape=(64, 64, 1), num_classes=2):
    """
    Minimal CNN for eye state classification.
    Input: 64×64 grayscale
    Output: open / closed
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.GlobalAveragePooling2D(),

        # Dense head
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax'),
    ], name='Eye_CNN')
    return model


def build_mouth_cnn(input_shape=(96, 96, 3), num_classes=2):
    """
    Minimal CNN for yawn detection.
    Input: 96×96 RGB
    Output: yawn / no-yawn
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.GlobalAveragePooling2D(),

        # Dense head
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax'),
    ], name='Mouth_CNN')
    return model


eye_model   = build_eye_cnn()
mouth_model = build_mouth_cnn()

eye_model.summary()
print('\n')
mouth_model.summary()

## 8. Data Augmentation

In [ ]:
# Light augmentation — enough to improve generalization without distorting facial features
eye_augmentor = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name='eye_aug')

mouth_augmentor = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
    layers.RandomRotation(0.05),
], name='mouth_aug')

print('Augmentors defined.')

## 9. Build tf.data Pipelines

In [ ]:
def make_dataset(X, y, augmentor=None, shuffle=True, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED)
    ds = ds.batch(batch_size)
    if augmentor is not None:
        ds = ds.map(
            lambda x, y: (augmentor(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )
    return ds.prefetch(tf.data.AUTOTUNE)


# Eye datasets
eye_train_ds = make_dataset(X_eye_tr,  y_eye_tr,  eye_augmentor)
eye_val_ds   = make_dataset(X_eye_val, y_eye_val, shuffle=False)
eye_test_ds  = make_dataset(X_eye_te,  y_eye_te,  shuffle=False)

# Mouth datasets
mo_train_ds = make_dataset(X_mo_tr,  y_mo_tr,  mouth_augmentor)
mo_val_ds   = make_dataset(X_mo_val, y_mo_val, shuffle=False)
mo_test_ds  = make_dataset(X_mo_te,  y_mo_te,  shuffle=False)

print('tf.data pipelines ready.')

## 10. Compile & Train — Eye CNN

In [ ]:
eye_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

eye_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
    callbacks.ModelCheckpoint(
        str(OUT / 'eye_model_best.h5'), monitor='val_accuracy',
        save_best_only=True, verbose=0
    ),
]

print('Training Eye CNN...')
eye_history = eye_model.fit(
    eye_train_ds,
    validation_data=eye_val_ds,
    epochs=EPOCHS,
    callbacks=eye_callbacks,
    verbose=1
)

## 11. Compile & Train — Mouth CNN

In [ ]:
mouth_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

mouth_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
    callbacks.ModelCheckpoint(
        str(OUT / 'mouth_model_best.h5'), monitor='val_accuracy',
        save_best_only=True, verbose=0
    ),
]

print('Training Mouth CNN...')
mouth_history = mouth_model.fit(
    mo_train_ds,
    validation_data=mo_val_ds,
    epochs=EPOCHS,
    callbacks=mouth_callbacks,
    verbose=1
)

## 12. Training Curves

In [ ]:
def plot_history(history, model_name, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} — Training Curves', fontsize=14, fontweight='bold')

    # Accuracy
    axes[0].plot(history.history['accuracy'],     label='Train Acc',  color='#3B82F6')
    axes[0].plot(history.history['val_accuracy'], label='Val Acc',    color='#F59E0B', linestyle='--')
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Loss
    axes[1].plot(history.history['loss'],     label='Train Loss', color='#EF4444')
    axes[1].plot(history.history['val_loss'], label='Val Loss',   color='#8B5CF6', linestyle='--')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(out_dir / f'{model_name}_curves.png', dpi=120)
    plt.show()


plot_history(eye_history,   'Eye_CNN',   OUT)
plot_history(mouth_history, 'Mouth_CNN', OUT)

## 13. Evaluation — Eye CNN

In [ ]:
def evaluate_model(model, X_test, y_test, class_names, model_name, out_dir):
    """
    Full evaluation: accuracy, precision, recall, F1, confusion matrix.
    """
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec  = recall_score(y_test, y_pred, average='weighted')
    f1   = f1_score(y_test, y_pred, average='weighted')

    print(f'\n{'='*50}')
    print(f'  {model_name} — Test Set Metrics')
    print(f'{'='*50}')
    print(f'  Accuracy : {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall   : {rec:.4f}')
    print(f'  F1 Score : {f1:.4f}')
    print(f'{'='*50}')
    print()
    print(classification_report(y_test, y_pred, target_names=class_names))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        linewidths=0.5, ax=ax
    )
    ax.set_title(f'{model_name} — Confusion Matrix', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.tight_layout()
    plt.savefig(out_dir / f'{model_name}_confusion.png', dpi=120)
    plt.show()

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}


eye_metrics = evaluate_model(
    eye_model, X_eye_te, y_eye_te,
    ['Open Eye', 'Closed Eye'], 'Eye_CNN', OUT
)

## 14. Evaluation — Mouth CNN

In [ ]:
mouth_metrics = evaluate_model(
    mouth_model, X_mo_te, y_mo_te,
    ['No Yawn', 'Yawn'], 'Mouth_CNN', OUT
)

## 15. NTHU Generalization Test

> We do **not** train on NTHU. It is used only to evaluate generalization across lighting, head-pose, and subject variation.

In [ ]:
# ── NTHU Eye Generalization Test ─────────────────────────────────────────────
# We do not directly train on NTHU due to its multi-modal nature;
# instead, it is used to evaluate generalization.

nthu_open_dir   = None
nthu_closed_dir = None

# Discover eye-related subfolders
try:
    all_dirs = list(NTHU_ROOT.rglob('*'))
    all_dirs = [d for d in all_dirs if d.is_dir()]

    nthu_open_dirs   = [d for d in all_dirs if 'open' in d.name.lower()]
    nthu_closed_dirs = [d for d in all_dirs if 'close' in d.name.lower()]

    print(f'NTHU open-eye folders found  : {len(nthu_open_dirs)}')
    print(f'NTHU closed-eye folders found: {len(nthu_closed_dirs)}')

    if nthu_open_dirs and nthu_closed_dirs:
        X_nthu_open,   _ = load_images_from_folder(nthu_open_dirs[0],   0, EYE_SIZE, 'grayscale', 500)
        X_nthu_closed, _ = load_images_from_folder(nthu_closed_dirs[0], 1, EYE_SIZE, 'grayscale', 500)

        X_nthu = np.concatenate([X_nthu_open, X_nthu_closed])
        y_nthu = np.concatenate([np.zeros(len(X_nthu_open)), np.ones(len(X_nthu_closed))]).astype(int)

        y_nthu_pred = np.argmax(eye_model.predict(X_nthu, verbose=0), axis=1)
        nthu_acc = accuracy_score(y_nthu, y_nthu_pred)

        print(f'\nNTHU Generalization Accuracy (Eye CNN): {nthu_acc:.4f}')
        print('→ Lower than in-distribution accuracy demonstrates need for temporal modeling.')
    else:
        print('Could not find matching NTHU eye folders — skipping NTHU eval.')
except Exception as e:
    print(f'NTHU evaluation skipped: {e}')

## 16. Class Activation Map (CAM) Visualization

In [ ]:
def make_gradcam(model, img, last_conv_name='conv2d_2'):
    """
    Simple GradCAM implementation to show what the CNN focuses on.
    """
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        x = tf.cast(img[np.newaxis, ...], tf.float32)
        conv_out, preds = grad_model(x)
        pred_class = tf.argmax(preds[0])
        class_score = preds[:, pred_class]

    grads    = tape.gradient(class_score, conv_out)[0]
    weights  = tf.reduce_mean(grads, axis=(0, 1))
    cam      = tf.reduce_sum(conv_out[0] * weights, axis=-1).numpy()
    cam      = np.maximum(cam, 0)
    cam      = cv2.resize(cam, (img.shape[1], img.shape[0]))
    cam      = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam, int(pred_class.numpy())


# Visualize GradCAM on 4 eye samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Eye CNN — GradCAM Activation Maps', fontsize=14, fontweight='bold')

sample_idx = np.random.choice(len(X_eye_te), 4, replace=False)
class_names_eye = ['Open', 'Closed']

for col, i in enumerate(sample_idx):
    img = X_eye_te[i]
    true_label = y_eye_te[i]

    try:
        cam, pred_label = make_gradcam(eye_model, img)
        # Original
        axes[0, col].imshow(img.squeeze(), cmap='gray')
        axes[0, col].set_title(f'True: {class_names_eye[true_label]}')
        axes[0, col].axis('off')
        # CAM
        axes[1, col].imshow(img.squeeze(), cmap='gray')
        axes[1, col].imshow(cam, alpha=0.5, cmap='jet')
        axes[1, col].set_title(f'Pred: {class_names_eye[pred_label]}')
        axes[1, col].axis('off')
    except:
        axes[0, col].axis('off')
        axes[1, col].axis('off')

plt.tight_layout()
plt.savefig(OUT / 'eye_gradcam.png', dpi=120)
plt.show()

## 17. Limitation Analysis — Why Frame-Based Fails

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║          CNN Baseline — Limitation Analysis                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║     Problem 1: No temporal context                               ║
║     A single closed-eye frame ≠ drowsiness.                      ║
║     A blink lasts ~150–400ms. Drowsy closure lasts 2–4 seconds.  ║
║     This CNN cannot distinguish them.                            ║
║                                                                  ║
║    Problem 2: Talking vs. yawning confusion                      ║
║     Open-mouth talking ≈ open-mouth yawning at the frame level.  ║
║     Duration and MAR trajectory are the real discriminators.     ║
║                                                                  ║
║     Problem 3: Illumination sensitivity                          ║
║     NTHU generalization gap reveals domain shift vulnerability.  ║
║                                                                  ║
║     Solution → Part B: PERCLOS + MAR temporal fusion             ║
║     Rolling window EAR analysis with MediaPipe Face Mesh         ║
║     captures the DURATION dimension the CNN ignores.             ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

## 18. Save Models & Summary Report

In [ ]:
# Save final models
eye_model.save(str(OUT / 'eye_cnn_final.h5'))
mouth_model.save(str(OUT / 'mouth_cnn_final.h5'))

# Summary table
summary = pd.DataFrame({
    'Model'    : ['Eye CNN (Open/Closed)', 'Mouth CNN (Yawn/No-Yawn)'],
    'Dataset'  : ['MRL Eye Dataset', 'Yawn Dataset'],
    'Accuracy' : [eye_metrics['accuracy'], mouth_metrics['accuracy']],
    'Precision': [eye_metrics['precision'], mouth_metrics['precision']],
    'Recall'   : [eye_metrics['recall'], mouth_metrics['recall']],
    'F1 Score' : [eye_metrics['f1'], mouth_metrics['f1']],
})

summary = summary.round(4)
summary.to_csv(OUT / 'cnn_baseline_results.csv', index=False)
print('\n=== CNN Baseline Results ===')
print(summary.to_string(index=False))

print(f'\nAll outputs saved to: {OUT}')
print('\nPart A complete. Proceed to Part B for the real-time temporal system.')